## Data Preprocessing

In [1]:
import os
import pandas as pd
import numpy as np
from PIL import Image
import random
from tqdm import tqdm
import cv2

In [2]:
from config import GLOBAL_CONFIG_DATA_PATH, GLOBAL_CONFIG_RESULT_PATH
GLOBAL_CONFIG_DATA_RAW_IMAGE_PATH = GLOBAL_CONFIG_DATA_PATH + 'Hist_exast/results/kapur/exhaustive/k4/'
GLOBAL_CONFIG_DATA_RAW_LABEL_PATH = GLOBAL_CONFIG_DATA_PATH + 'Train_kapur.csv'

In [3]:
# resize image to target shape
def resize_image(img, target_shape):
    return cv2.resize(img, target_shape, interpolation=cv2.INTER_NEAREST)

In [4]:
# Set paths
image_dir = GLOBAL_CONFIG_DATA_RAW_IMAGE_PATH  # Folder with original images (kapur)
base_image_dir = GLOBAL_CONFIG_DATA_PATH + 'images/'  # Folder with original images (Unprocessed)
output_dir = GLOBAL_CONFIG_DATA_PATH + 'cropped_images/kapur/'  # Folder to save cropped images
output_dir_base = GLOBAL_CONFIG_DATA_PATH + 'cropped_images/base/'  # Folder to save cropped images
output_csv_path = GLOBAL_CONFIG_DATA_PATH + 'kapur_cropped_train.csv'  # Path to save new CSV

# Load CSV data
csv_path = GLOBAL_CONFIG_DATA_RAW_LABEL_PATH
df = pd.read_csv(csv_path)

# print the class distribution
print(df['class'].value_counts())


# split df into 3 dfs based on class
df_Trophozoite = df[df['class'] == 'Trophozoite']
df_WBC = df[df['class'] == 'WBC']
df_NEG = df[df['class'] == 'NEG']

# duplicate All Rows with class 'NEG' 22 times to balance the data
df_NEG = pd.concat([df_NEG]*22, ignore_index=True)

# duplicate all rows with class 'WBC' to a seperate df to balance the data and add random error to the bounding box +- 10%
df_WBC_Deep_Copy = df_WBC.copy(deep=True)

# add random error to the bounding box +- 10%
max_pixel_error = max(df_WBC['xmax'].max() - df_WBC['xmin'].min(), df_WBC['ymax'].max() - df_WBC['ymin'].min()) * 0.1

df_WBC.loc[:, 'xmin'] += np.random.randint(-max_pixel_error, max_pixel_error, df_WBC.shape[0])
df_WBC.loc[:, 'ymin'] += np.random.randint(-max_pixel_error, max_pixel_error, df_WBC.shape[0])
df_WBC.loc[:, 'xmax'] += np.random.randint(-max_pixel_error, max_pixel_error, df_WBC.shape[0])
df_WBC.loc[:, 'ymax'] += np.random.randint(-max_pixel_error, max_pixel_error, df_WBC.shape[0])


# concat the original diff dfs into one
df = pd.concat([df_Trophozoite, df_WBC, df_WBC_Deep_Copy, df_NEG], ignore_index=True)

# print the class distribution
print(df['class'].value_counts())


# Create output directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)
os.makedirs(output_dir_base, exist_ok=True)

# Function to get a random bounding box location within image bounds
def get_random_bbox(img_size, box_size):
    max_y = img_size[1] - box_size[1]
    max_x = img_size[0] - box_size[0]
    ymin = random.randint(0, max_y)
    xmin = random.randint(0, max_x)
    return ymin, xmin, ymin + box_size[1], xmin + box_size[0]

# Calculate the largest bounding box dimensions
largest_box_width = df['xmax'] - df['xmin']
largest_box_height = df['ymax'] - df['ymin']
max_width = largest_box_width.max()
max_height = largest_box_height.max()
avg_width = largest_box_width.mean()
avg_height = largest_box_height.mean()
max_box_size = (150, 150)

# claculate disk space needed for the cropped images
disk_space_needed = (df.shape[0] * max_box_size[0] * max_box_size[1] * 3 / 1024 / 1024)*2
print(f"Disk space needed for cropped images: {disk_space_needed:.2f} MB")

print(f"Largest bounding box width: {max_width}")
print(f"Largest bounding box height: {max_height}")
print(f"Average bounding box width: {avg_width}")
print(f"Average bounding box height: {avg_height}")

# Prepare new CSV records
new_records = []

# Process each image
for idx, row in tqdm(df.iterrows()):
    img_path = os.path.join(image_dir, row['Image_ID'])

    # get base image id (split id at '_k4_' and take the first part)
    base_img_id = row['Image_ID'].split('_k4_')[0] + '.jpg'
    base_img_path = os.path.join(base_image_dir, base_img_id)

    try:
        with Image.open(img_path) as img, Image.open(base_img_path) as base_img:
            img = img.convert('RGB')
            img_width, img_height = img.size

            base_img = base_img.convert('RGB')
            base_img_width, base_img_height = base_img.size

            # Crop using bounding box or random box for 'NEG' classifications
            if row['class'] == 'NEG':
                ymin, xmin, ymax, xmax = get_random_bbox((img_width, img_height), max_box_size)
            else:
                ymin, xmin, ymax, xmax = row['ymin'], row['xmin'], row['ymax'], row['xmax']

            # Crop and resize to largest bounding box size
            
            # if Coordinate 'right' is less than 'left'
            if xmin > xmax:
                xmin, xmax = xmax, xmin
            # if Coordinate 'bottom' is less than 'top'
            if ymin > ymax:
                ymin, ymax = ymax, ymin

            cropped_img = img.crop((xmin, ymin, xmax, ymax))
            cropped_img = cropped_img.resize(max_box_size, Image.LANCZOS)  # Replaced ANTIALIAS with LANCZOS

            # Crop and resize base image to largest bounding box size
            cropped_base_img = base_img.crop((xmin, ymin, xmax, ymax))
            cropped_base_img = cropped_base_img.resize(max_box_size, Image.LANCZOS)  # Replaced ANTIALIAS with LANCZOS

            # Save the cropped image
            cropped_img_id = f"k_cropped_{idx}_{row['Image_ID']}"
            cropped_img_path = os.path.join(output_dir, cropped_img_id)
            cropped_img.save(cropped_img_path)

            # Save the cropped base image
            cropped_base_img_id = f"b_cropped_{idx}_{base_img_id}"
            cropped_base_img_path = os.path.join(output_dir_base, cropped_base_img_id)
            cropped_base_img.save(cropped_base_img_path)

            # Append new record to new CSV data
            new_record = {
                'cropped_Image_ID': cropped_img_id,
                'cropped_base_Image_ID': cropped_base_img_id,
                'width': max_box_size[0],
                'height': max_box_size[1],
                'class': row['class'],
                'confidence': row['confidence'],
                'ymin': ymin,
                'xmin': xmin,
                'ymax': ymax,
                'xmax': xmax,
                'original_Image_ID': row['Image_ID'],
                'base_Image_ID': base_img_id
            }
            new_records.append(new_record)

            # save csv every 1000 images
            if idx % 1000 == 0:
                new_df = pd.DataFrame(new_records)
                new_df.to_csv(output_csv_path, index=False)

    except FileNotFoundError:
        print(f"Image {row['Image_ID']} not found. Skipping.")

# Save new CSV with cropped image data
new_df = pd.DataFrame(new_records)
new_df.to_csv(output_csv_path, index=False)
print("Cropped images saved and new CSV created.")


class
Trophozoite    15838
WBC             7004
NEG              688
Name: count, dtype: int64
class
Trophozoite    15838
NEG            15136
WBC            14008
Name: count, dtype: int64
Disk space needed for cropped images: 5791.25 MB
Largest bounding box width: 1043
Largest bounding box height: 1121
Average bounding box width: 55.33664576941888
Average bounding box height: 55.21679783024321


44982it [1:05:21, 11.47it/s]


Cropped images saved and new CSV created.


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import cv2
from pathlib import Path
import matplotlib.pyplot as plt

class MalariaClassifier:
    def __init__(self, input_shape=(150, 150, 3), num_classes=3):
        # Set mixed precision policy
        tf.keras.mixed_precision.set_global_policy('mixed_float16')
        self.input_shape = input_shape
        self.num_classes = num_classes
        self.model = self.build_model()
        self.label_encoder = LabelEncoder()
        
    def build_model(self):
        """Build the CNN architecture with mixed precision."""
        model = models.Sequential([
            # First Convolutional Block
            layers.Conv2D(32, (3, 3), activation='relu', input_shape=self.input_shape, dtype='float16'),
            layers.BatchNormalization(),
            layers.MaxPooling2D((2, 2)),
            layers.Dropout(0.25),
            
            # Second Convolutional Block
            layers.Conv2D(64, (3, 3), activation='relu', dtype='float16'),
            layers.BatchNormalization(),
            layers.MaxPooling2D((2, 2)),
            layers.Dropout(0.25),
            
            # Third Convolutional Block
            layers.Conv2D(128, (3, 3), activation='relu', dtype='float16'),
            layers.BatchNormalization(),
            layers.MaxPooling2D((2, 2)),
            layers.Dropout(0.25),
            
            # Dense Layers
            layers.Flatten(),
            layers.Dense(256, activation='relu', dtype='float16'),
            layers.BatchNormalization(),
            layers.Dropout(0.5),
            layers.Dense(self.num_classes, activation='softmax', dtype='float32')  # Final layer stays float32
        ])
        
        # Compile model with mixed precision optimizer
        optimizer = tf.keras.optimizers.Adam()
        optimizer = tf.keras.mixed_precision.LossScaleOptimizer(optimizer)
        
        model.compile(
            optimizer=optimizer,
            loss='categorical_crossentropy',
            metrics=['accuracy']
        )
        
        return model
    
    def load_and_preprocess_image(self, image_path):
        """Load and preprocess a single image with float16."""
        img = cv2.imread(str(image_path))
        if img is None:
            raise ValueError(f"Could not load image: {image_path}")
            
        # Resize to match input shape
        img = cv2.resize(img, (self.input_shape[0], self.input_shape[1]))
        
        # Convert to float16 and normalize
        img = img.astype(np.float16) / 255.0
        return img
    
    def prepare_data(self, csv_path, image_dir, batch_size=32):
        """Prepare data for training using generators to save memory."""
        df = pd.read_csv(csv_path)
        
        # Create train/validation split
        train_df, val_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['class'])
        
        # Fit label encoder on all classes
        self.label_encoder.fit(df['class'].values)
        
        # Create data generators
        train_gen = self.data_generator(train_df, image_dir, batch_size)
        val_gen = self.data_generator(val_df, image_dir, batch_size)
        
        return train_gen, val_gen, len(train_df), len(val_df)
    
    def data_generator(self, df, image_dir, batch_size):
        """Generator function to load images in batches."""
        num_samples = len(df)
        while True:
            # Shuffle the DataFrame at the start of each epoch
            df = df.sample(frac=1).reset_index(drop=True)
            
            for offset in range(0, num_samples, batch_size):
                batch_df = df.iloc[offset:min(offset + batch_size, num_samples)]
                
                # Initialize batch arrays
                X_batch = np.zeros((len(batch_df), *self.input_shape), dtype=np.float16)
                y_batch = np.zeros((len(batch_df), self.num_classes), dtype=np.float16)
                
                for idx, (_, row) in enumerate(batch_df.iterrows()):
                    # Load and preprocess image
                    img_path = Path(image_dir) / row['cropped_Image_ID']
                    try:
                        X_batch[idx] = self.load_and_preprocess_image(img_path)
                        
                        # Convert label to one-hot encoding
                        label_idx = self.label_encoder.transform([row['class']])[0]
                        y_batch[idx] = tf.keras.utils.to_categorical(label_idx, self.num_classes)
                    except Exception as e:
                        print(f"Error processing {img_path}: {str(e)}")
                        continue
                
                yield X_batch, y_batch
    
    def train(self, train_gen, val_gen, train_steps, val_steps, epochs=50):
        """Train the model using generators."""
        # Create callbacks
        early_stopping = tf.keras.callbacks.EarlyStopping(
            monitor='val_loss',
            patience=10,
            restore_best_weights=True
        )
        
        reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.2,
            patience=5,
            min_lr=1e-6
        )
        
        # Train model
        history = self.model.fit(
            train_gen,
            steps_per_epoch=train_steps,
            epochs=epochs,
            validation_data=val_gen,
            validation_steps=val_steps,
            callbacks=[early_stopping, reduce_lr]
        )
        
        return history
    
    def plot_training_history(self, history):
        """Plot training history."""
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
        
        # Plot accuracy
        ax1.plot(history.history['accuracy'])
        ax1.plot(history.history['val_accuracy'])
        ax1.set_title('Model Accuracy')
        ax1.set_ylabel('Accuracy')
        ax1.set_xlabel('Epoch')
        ax1.legend(['Train', 'Validation'])
        
        # Plot loss
        ax2.plot(history.history['loss'])
        ax2.plot(history.history['val_loss'])
        ax2.set_title('Model Loss')
        ax2.set_ylabel('Loss')
        ax2.set_xlabel('Epoch')
        ax2.legend(['Train', 'Validation'])
        
        plt.show()

def main():
    # Configuration
    image_dir = f"{GLOBAL_CONFIG_DATA_PATH}/cropped_images/kapur/"
    csv_path = f"{GLOBAL_CONFIG_DATA_PATH}/kapur_cropped_train.csv"
    batch_size = 32
    
    # Initialize classifier
    classifier = MalariaClassifier(input_shape=(100, 100, 3), num_classes=3)
    
    # Prepare data generators
    print("Setting up data generators...")
    train_gen, val_gen, train_samples, val_samples = classifier.prepare_data(
        csv_path, 
        image_dir, 
        batch_size=batch_size
    )
    
    # Calculate steps per epoch
    train_steps = train_samples // batch_size
    val_steps = val_samples // batch_size
    
    print(f"Training samples: {train_samples}")
    print(f"Validation samples: {val_samples}")
    
    # Train model
    print("\nTraining model...")
    history = classifier.train(
        train_gen,
        val_gen,
        train_steps,
        val_steps,
        epochs=50
    )
    
    # Plot training history
    classifier.plot_training_history(history)
    
    # Save model
    model_save_path = f"{GLOBAL_CONFIG_DATA_PATH}/models/malaria_classifier.h5"
    classifier.model.save(model_save_path)
    print(f"\nModel saved to {model_save_path}")

if __name__ == "__main__":
    main()

e:\Uni Projects\COS 711\Repo\COS711_Assingment3\venv\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Setting up data generators...
Training samples: 35985
Validation samples: 8997

Training model...
Epoch 1/50
1124/1124 ━━━━━━━━━━━━━━━━━━━━ 1506s 1s/step - accuracy: 0.7639 - loss: 0.6762 - val_accuracy: 0.4912 - val_loss: 22.5379 - learning_rate: 0.0010
Epoch 2/50
1124/1124 ━━━━━━━━━━━━━━━━━━━━ 1508s 1s/step - accuracy: 0.8495 - loss: 0.4002 - val_accuracy: 0.6708 - val_loss: 33.2076 - learning_rate: 0.0010
Epoch 3/50
1124/1124 ━━━━━━━━━━━━━━━━━━━━ 1492s 1s/step - accuracy: 0.8720 - loss: 0.3492 - val_accuracy: 0.8958 - val_loss: 0.3707 - learning_rate: 0.0010
Epoch 4/50
1124/1124 ━━━━━━━━━━━━━━━━━━━━ 1504s 1s/step - accuracy: 0.8822 - loss: 0.3255 - val_accuracy: 0.6052 - val_loss: 1.1323 - learning_rate: 0.0010
Epoch 5/50
1124/1124 ━━━━━━━━━━━━━━━━━━━━ 1473s 1s/step - accuracy: 0.8883 - loss: 0.3153 - val_accuracy: 0.6711 - val_loss: 0.5788 - learning_rate: 0.0010
Epoch 6/50
1124/1124 ━━━━━━━━━━━━━━━━━━━━ 1511s 1s/step - accuracy: 0.8931 - loss: 0.3056 - val_accuracy: 0.9052 - val_l

In [ ]:
# Assuming `early_stopping` is the callback used during training
classifier.model.set_weights(early_stopping.best_weights)
